# Level 4, Notebook 3: Fast and safe caching

**Problems in this notebook**

- Problem 33: Build a Simple LRU Cache for Prompts
- Problem 37: Tenant-Aware Caching

**How to use this notebook**

1. Read the problem and say your plan aloud before you write any code.
2. Use the AI prompt given (or your own narrow one) to fill the starter. Read the AI's code before you run it.
3. Run the checks. Open the reference only after you have tried.

The ideas behind these problems are in `03_safe_caching_THEORY.md` in this folder.

In [ ]:
def run_checks(fn, cases):
    """cases: list of (args_tuple, expected). Prints a short report, never raises."""
    passed = 0
    for i, (args, expected) in enumerate(cases, 1):
        try:
            got = fn(*args)
        except NotImplementedError:
            print("Not implemented yet. Fill in the starter first.")
            return False
        except Exception as e:
            print(f"Case {i}: raised {type(e).__name__}: {e}")
            continue
        if got == expected:
            passed += 1
        else:
            print(f"Case {i}: expected {expected!r}, got {got!r}")
    print(f"{passed}/{len(cases)} checks passed")
    return passed == len(cases)

## Problem 33: Build a Simple LRU Cache for Prompts

**The situation**

Rendering a prompt template and fetching its context takes time and money. A small in-memory cache keeps the most recent results so repeat requests are fast.

**Your task**

Build `PromptCache(capacity)` with `get(key)` and `put(key, value)`. When full, drop the least recently used item. Reading an item counts as using it. A capacity of zero or less raises `ValueError`.

**Example**

Capacity 1: `put("a", 1)`, `put("b", 2)`, then `get("a")` gives `None` and `get("b")` gives `2`.

**Ask before coding**

- What capacity should it start with?
- Should reading an item make it "recent"?
- Do entries need to expire after some time?

**Plan to say aloud**

1. Store items in an `OrderedDict`, oldest first.
2. On `get`, return `None` if missing; otherwise move the key to the end and return the value.
3. On `put`, write the value and move the key to the end.
4. If the size is over capacity, remove the first (oldest) item.

### Prompt for your AI assistant

```text
Write a Python class PromptCache with __init__(self, capacity: int), get(self, key: str) and put(self, key: str, value) -> None. It is an LRU cache built on collections.OrderedDict. Raise ValueError if capacity <= 0. get returns None for a missing key and marks a found key as most recently used. put inserts or updates, marks the key as most recent, and evicts the least recently used item when over capacity. Show only the class.
```

**Check the AI's code for**

- Does not move a key to the end on `get`, so it evicts by insertion order (FIFO), not by use.
- Updating an existing key leaves it in its old position, so a fresh value can be evicted next.
- Evicts before inserting, which drops an item even when the key already existed.
- Uses `popitem()` with the default `last=True`, which removes the newest item instead of the oldest.

In [ ]:
class PromptCache:
    """Least-recently-used cache.

    Rules:
    - capacity <= 0 -> ValueError
    - get(key): None if missing; otherwise mark as most recent and return value
    - put(key, value): insert or update, mark as most recent
    - over capacity -> remove the least recently used item
    """

    def __init__(self, capacity: int):
        raise NotImplementedError

    def get(self, key: str):
        raise NotImplementedError

    def put(self, key: str, value) -> None:
        raise NotImplementedError

In [ ]:
def replay_cache(cache_class, capacity, ops):
    """ops: ("put", key, value) or ("get", key). Returns the list of get results."""
    try:
        cache = cache_class(capacity)
    except ValueError:
        return "ValueError"
    results = []
    for op in ops:
        if op[0] == "put":
            cache.put(op[1], op[2])
        else:
            results.append(cache.get(op[1]))
    return results

cases = [
    ((1, [("put", "a", 1), ("put", "b", 2), ("get", "a"), ("get", "b")]), [None, 2]),
    ((2, [("put", "a", 1), ("put", "b", 2), ("get", "a"), ("put", "c", 3),
          ("get", "b"), ("get", "a"), ("get", "c")]), [1, None, 1, 3]),        # reading "a" saved it
    ((2, [("put", "a", 1), ("put", "b", 2), ("put", "a", 10), ("put", "c", 3),
          ("get", "a"), ("get", "b")]), [10, None]),                           # update refreshes "a"
    ((2, [("get", "missing")]), [None]),
    ((0, []), "ValueError"),
]
run_checks(lambda *a: replay_cache(PromptCache, *a), cases)

### Reference solution

Try first. Then compare.

In [ ]:
from collections import OrderedDict

class PromptCacheReference:
    def __init__(self, capacity: int):
        if capacity <= 0:
            raise ValueError("capacity must be positive")
        self.capacity = capacity
        self.data = OrderedDict()

    def get(self, key: str):
        if key not in self.data:
            return None
        self.data.move_to_end(key)
        return self.data[key]

    def put(self, key: str, value) -> None:
        self.data[key] = value
        self.data.move_to_end(key)
        if len(self.data) > self.capacity:
            self.data.popitem(last=False)

assert run_checks(lambda *a: replay_cache(PromptCacheReference, *a), cases)

**Explain it in 30 seconds**

> "I use an OrderedDict, which remembers order. Every time a key is read or written I move it to the end, so the front always holds the item nobody has touched for longest. When the cache grows past its capacity I remove the front item. A zero capacity is a config mistake, so I raise an error."

**Edge cases to mention**

- Capacity zero or negative.
- Putting a key that already exists (update, not a second copy).
- Two tenants sending the same key text, which must not share an entry.
- Stale entries after a prompt template changes.

**If they ask "how would this work in production?"**

- Several servers need one shared cache, such as Redis, with a time-to-live on each entry.
- Put tenant, permission version, model, and template version in the key (Problem 37).
- Do not cache answers that contain sensitive data, and watch the hit rate to see if the cache earns its memory.

## Problem 37: Tenant-Aware Caching

**The situation**

Two companies both ask "What is our refund policy?". Their documents are different, so a cached answer for one must never be served to the other. The same goes for two users with different access rights.

**Your task**

Write `tenant_aware_cache_key(tenant_id, user_id, query, permission_version="v1")`. Build the key `tenant:user:permission_version:normalized_query`. Return `None` (do not cache) if the tenant or user is missing or contains `:`.

**Example**

`("t1", "u1", " Hello  World ")` gives `"t1:u1:v1:hello world"`. `("", "u1", "hi")` gives `None`.

**Ask before coding**

- Should the key include the user, or is the tenant enough?
- Is there a permission version that changes when access rights change?
- Should model and index versions be part of the key too?

**Plan to say aloud**

1. Check tenant and user are present and contain no `:`. If not, return `None`.
2. Normalize the query: lowercase, trim, and squash runs of whitespace into one space.
3. Join tenant, user, permission version, and query with `:`.

### Prompt for your AI assistant

```text
Write a Python function tenant_aware_cache_key(tenant_id: str, user_id: str, query: str, permission_version: str = 'v1') -> str | None. Return None if tenant_id or user_id is empty, only whitespace, or contains ':'. Normalize the query: strip, lowercase, and collapse any whitespace (spaces, tabs, newlines) into a single space. Return f'{tenant_id}:{user_id}:{permission_version}:{normalized}'. Show only the function.
```

**Check the AI's code for**

- Builds the key from the query alone, which shares answers across tenants.
- Leaves out the permission version, so a user who just lost access still gets the old answer.
- Replaces only single spaces (`query.replace("  ", " ")`) and misses tabs and newlines.
- Does not reject `:` in ids, so tenant `"t1:u1"` with user `"x"` could collide with tenant `"t1"` and user `"u1:x"`.

In [ ]:
def tenant_aware_cache_key(tenant_id: str, user_id: str, query: str, permission_version: str = "v1") -> str | None:
    """Build a cache key that cannot be shared across tenants or users.

    Rules:
    - tenant_id or user_id empty, blank, or containing ":" -> None (do not cache)
    - normalize query: strip, lowercase, collapse all whitespace to one space
    - return "tenant:user:permission_version:query"
    """
    raise NotImplementedError

In [ ]:
cases = [
    (("t1", "u1", " Hello  World "), "t1:u1:v1:hello world"),
    (("t1", "u1", "Refund Policy", "v7"), "t1:u1:v7:refund policy"),
    (("t1", "u1", "refund\tpolicy\n"), "t1:u1:v1:refund policy"),  # tabs and newlines
    (("t2", "u1", "Refund Policy"), "t2:u1:v1:refund policy"),     # other tenant, different key
    (("", "u1", "hi"), None),                                      # no tenant: do not cache
    (("t1:u1", "x", "hi"), None),                                  # separator inside an id
]
run_checks(tenant_aware_cache_key, cases)

### Reference solution

Try first. Then compare.

In [ ]:
def tenant_aware_cache_key_reference(tenant_id: str, user_id: str, query: str, permission_version: str = "v1") -> str | None:
    for part in (tenant_id, user_id):
        if not part or not part.strip() or ":" in part:
            return None
    normalized = " ".join(query.lower().split())
    return f"{tenant_id}:{user_id}:{permission_version}:{normalized}"

assert run_checks(tenant_aware_cache_key_reference, cases)

**Explain it in 30 seconds**

> "A cache hit is only safe if the answer was made for the same tenant, the same user, and the same access rights. So all three go into the key, followed by the cleaned-up query. If the tenant or user is missing I return None, which means skip the cache. Bumping the permission version makes old entries unreachable."

**Edge cases to mention**

- Missing tenant or user id.
- A `:` inside an id, which could make two different keys look the same.
- Permissions changed, so old answers must not be served.
- A long query with personal data ending up in the key.

**If they ask "how would this work in production?"**

- Hash the normalized query (for example SHA-256) so raw user text is not stored in cache keys or logs.
- Add model, prompt template, and index version so a re-index or model change does not serve stale answers.
- Set a time-to-live, and skip caching for answers that contain sensitive data.

## Wrap-up

- A cache trades memory for speed, so it needs a size limit and a rule for what to drop.
- In a multi-tenant system the key decides who can see an entry. Tenant, user, and permission version belong in it.
- When you cannot build a safe key, skip the cache. A slow answer is better than another customer's answer.